# 04 — Retrain IsolationForest (Anomaly) cho fair comparison vs CNN1D

**Mục tiêu:** train lại `IsolationForest` với protocol đúng cho anomaly detection để có bảng so sánh anomaly vs supervised có ý nghĩa khoa học.

**Vấn đề với model cũ** (`models/isolation_forest.pkl` AUC=0.5306):
- Train trên cả 67 features (nhiều feature không liên quan exfil)
- `contamination=0.05` quá cao so với tỷ lệ exfil thực 0.08%
- Train trên cả normal + exfil (anomaly model nên train CHỈ trên normal)

**Cải thiện:**
1. Chọn 12 features upload-related (giảm noise)
2. Train chỉ trên NORMAL rows
3. `contamination=0.001` ≈ tỷ lệ exfil thực
4. Evaluate trên cùng test set với CNN1D để so sánh fair

**Output:** `models/isolation_forest_v2.pkl`, `models/scaler_anomaly_v2.pkl`, `data/processed/anomaly_results_v2.json`

---

## Cách chạy trên Google Colab

1. Upload thư mục `data/processed/` (chỉ cần `train.csv`, `test.csv`, `val.csv`, `feature_cols.json`) lên Google Drive.
2. Chạy cell `Mount Drive` rồi sửa `DRIVE_BASE` trỏ tới folder đã upload.
3. Run All. Kết quả tải về local thông qua cell cuối.

## Cách chạy local

Bỏ qua cell `Mount Drive`, đặt `DRIVE_BASE = Path('../data/processed')` và chạy từ Jupyter trong env `exfil`.

## 1. Setup

In [ ]:
# Colab only — uncomment 2 cell duoi neu chay tren Colab
# from google.colab import drive
# drive.mount('/content/drive')

# !pip -q install scikit-learn==1.5.* joblib pandas numpy matplotlib seaborn

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score,
    confusion_matrix, precision_recall_curve,
)
import joblib

# === SUA DUONG DAN O DAY ===
# Colab:  DRIVE_BASE = Path('/content/drive/MyDrive/Exfiltration/data/processed')
# Local:  DRIVE_BASE = Path('../data/processed')
DRIVE_BASE = Path('../data/processed')
MODEL_OUT  = Path('../models')
RESULTS_OUT = DRIVE_BASE  # luu json vao day
MODEL_OUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print('DRIVE_BASE =', DRIVE_BASE.resolve())

## 2. Load data

In [ ]:
train_df = pd.read_csv(DRIVE_BASE / 'train.csv')
test_df  = pd.read_csv(DRIVE_BASE / 'test.csv')
val_df   = pd.read_csv(DRIVE_BASE / 'val.csv')

with open(DRIVE_BASE / 'feature_cols.json') as f:
    all_feature_cols = json.load(f)

print(f'train: {train_df.shape}, exfil={train_df.exfil_label.sum()} ({train_df.exfil_label.mean()*100:.3f}%)')
print(f'test:  {test_df.shape},  exfil={test_df.exfil_label.sum()} ({test_df.exfil_label.mean()*100:.3f}%)')
print(f'val:   {val_df.shape},   exfil={val_df.exfil_label.sum()} ({val_df.exfil_label.mean()*100:.3f}%)')
print(f'total features: {len(all_feature_cols)}')

## 3. Chon feature subset upload-related

Lap luan chon feature: exfiltration co dac trung **upload >> download**, **flow ngan** (automated), **PSH flag cao** (active transfer), **packet rate cao** (burst).

12 feature duoc chon:

In [ ]:
EXFIL_FEATURES = [
    # Upload signal
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'Avg Fwd Segment Size',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',
    # Up/Down asymmetry
    'Down/Up Ratio',
    # Burst / rate
    'Fwd Packets/s',
    'Flow Bytes/s',
    'Flow IAT Std',
    'Flow IAT Mean',
    # Flag indicators (active transfer)
    'PSH Flag Count',
    'ACK Flag Count',
]

missing = [c for c in EXFIL_FEATURES if c not in train_df.columns]
assert not missing, f'Thieu cot: {missing}'
print('OK — co du', len(EXFIL_FEATURES), 'feature')

## 4. Tach normal/exfil va fit scaler

In [ ]:
X_train_full = train_df[EXFIL_FEATURES].values
y_train_full = train_df['exfil_label'].values

X_test  = test_df[EXFIL_FEATURES].values
y_test  = test_df['exfil_label'].values

X_val   = val_df[EXFIL_FEATURES].values
y_val   = val_df['exfil_label'].values

# Replace inf, fillna 0
for arr in [X_train_full, X_test, X_val]:
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

# Anomaly model: train CHI tren NORMAL rows
X_train_normal = X_train_full[y_train_full == 0]
print(f'Train normal-only: {X_train_normal.shape[0]:,} rows')

# Fit scaler tren normal-only
scaler = StandardScaler()
X_train_normal_s = scaler.fit_transform(X_train_normal)
X_test_s = scaler.transform(X_test)
X_val_s  = scaler.transform(X_val)

## 5. Train IsolationForest (anomaly v2)

In [ ]:
EXFIL_RATE_TRAIN = float(y_train_full.mean())
print(f'Real exfil rate trong train: {EXFIL_RATE_TRAIN*100:.4f}%')

# Contamination = ti le exfil thuc te (xap xi)
CONTAMINATION = max(EXFIL_RATE_TRAIN, 1e-4)
print(f'Contamination su dung: {CONTAMINATION:.4f}')

t0 = time.time()
iforest = IsolationForest(
    n_estimators=300,
    contamination=CONTAMINATION,
    max_samples='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iforest.fit(X_train_normal_s)
train_time = time.time() - t0
print(f'Train xong sau {train_time:.1f}s')

## 6. Evaluate tren cung test set voi CNN1D

Higher score = more anomalous = more exfil → flip dau decision_function.

In [ ]:
t0 = time.time()
raw_test  = iforest.decision_function(X_test_s)
scores_test = -raw_test  # high = exfil
raw_val   = iforest.decision_function(X_val_s)
scores_val = -raw_val
infer_time = time.time() - t0
print(f'Inference time: {infer_time*1000:.1f}ms cho {len(X_test_s) + len(X_val_s):,} flows')
print(f'Per-sample: {infer_time*1e6/(len(X_test_s)+len(X_val_s)):.2f}us')

In [ ]:
# AUC tren test
auc_test = roc_auc_score(y_test, scores_test)
auc_val  = roc_auc_score(y_val,  scores_val)
print(f'AUC-ROC test: {auc_test:.4f}')
print(f'AUC-ROC val:  {auc_val:.4f}')

# Tuning threshold tren val: chon nguong sao cho FPR <= 5% va recall max
fpr_v, tpr_v, thr_v = roc_curve(y_val, scores_val)
# Lay nguong dau tien co FPR <= 0.05
candidates = np.where(fpr_v <= 0.05)[0]
if len(candidates) == 0:
    optimal_thr = thr_v[np.argmax(tpr_v - fpr_v)]  # Youden
else:
    best_idx = candidates[np.argmax(tpr_v[candidates])]
    optimal_thr = thr_v[best_idx]
print(f'Optimal threshold (FPR<=5%, recall max): {optimal_thr:.6f}')

In [ ]:
# Apply threshold tren test
preds_test = (scores_test >= optimal_thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, preds_test, labels=[0, 1]).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
metrics = {
    'auc_roc_test': float(auc_test),
    'auc_roc_val':  float(auc_val),
    'optimal_threshold': float(optimal_thr),
    'precision': float(precision_score(y_test, preds_test, zero_division=0)),
    'recall':    float(recall_score(y_test, preds_test, zero_division=0)),
    'f1':        float(f1_score(y_test, preds_test, zero_division=0)),
    'fpr':       float(fpr),
    'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    'train_time_sec': float(train_time),
    'inference_time_per_sample_us': float(infer_time*1e6/(len(X_test_s)+len(X_val_s))),
    'features_used': EXFIL_FEATURES,
    'contamination': float(CONTAMINATION),
    'n_train_normal': int(X_train_normal.shape[0]),
    'model_name': 'IsolationForest_v2',
    'model_path': 'models/isolation_forest_v2.pkl',
}
print(json.dumps(metrics, indent=2))

## 7. Plot ROC curve

In [ ]:
fpr_t, tpr_t, _ = roc_curve(y_test, scores_test)
plt.figure(figsize=(8, 6))
plt.plot(fpr_t, tpr_t, label=f'IsolationForest v2 (AUC={auc_test:.4f})', linewidth=2, color='#3498db')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
plt.axvline(0.05, color='red', linestyle=':', alpha=0.5, label='FPR=5%')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC — IsolationForest v2 (anomaly, upload features only)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_BASE / '..' / '..' / 'notebooks' / 'roc_isolation_forest_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save model + metadata

In [ ]:
joblib.dump(iforest, MODEL_OUT / 'isolation_forest_v2.pkl')
joblib.dump(scaler,  MODEL_OUT / 'scaler_anomaly_v2.pkl')

metadata = {
    'output': str(MODEL_OUT / 'isolation_forest_v2.pkl'),
    'scaler': str(MODEL_OUT / 'scaler_anomaly_v2.pkl'),
    'feature_names': EXFIL_FEATURES,
    'metrics': metrics,
}
with open(MODEL_OUT / 'isolation_forest_v2.json', 'w') as f:
    json.dump(metadata, f, indent=2)

with open(RESULTS_OUT / 'anomaly_results_v2.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved:')
print(f'  {MODEL_OUT / "isolation_forest_v2.pkl"}')
print(f'  {MODEL_OUT / "scaler_anomaly_v2.pkl"}')
print(f'  {MODEL_OUT / "isolation_forest_v2.json"}')
print(f'  {RESULTS_OUT / "anomaly_results_v2.json"}')

## 9. Sanity check — test tren val set

In [ ]:
preds_val = (scores_val >= optimal_thr).astype(int)
tn_v, fp_v, fn_v, tp_v = confusion_matrix(y_val, preds_val, labels=[0,1]).ravel()
print('VAL set:')
print(f'  AUC: {auc_val:.4f}')
print(f'  Recall: {recall_score(y_val, preds_val, zero_division=0):.4f}')
print(f'  FPR: {fp_v/(fp_v+tn_v):.4f}')
print(f'  TP={tp_v}, FP={fp_v}, TN={tn_v}, FN={fn_v}')

## 10. (Colab) tai file ve local

Sau khi chay xong, neu o Colab thi uncomment cell duoi de download model file ve laptop:

In [ ]:
# Colab only:
# from google.colab import files
# files.download(str(MODEL_OUT / 'isolation_forest_v2.pkl'))
# files.download(str(MODEL_OUT / 'scaler_anomaly_v2.pkl'))
# files.download(str(MODEL_OUT / 'isolation_forest_v2.json'))

---

## Tieu chi dat

- **AUC test >= 0.70**: bang so sanh co y nghia.
- **Recall test >= 0.30** voi FPR <= 5%: detect duoc mot phan exfil.
- **Inference time < 100us/flow**: phu hop real-time.

Neu khong dat AUC >= 0.7:
1. Thu them feature `Init_Win_bytes_forward`, `Init_Win_bytes_backward`.
2. Thu `OneClassSVM(nu=0.001)` tren subsample 50K normal.
3. Bao Cao van trinh bay ket qua nay nhu *bang chung dinh luong* rang anomaly thuan kho voi exfil tren CICIDS — la ket luan khoa hoc hop le.